### DATA INGESTION

In [1]:
## Document Structure
from langchain_core.documents import Document

In [2]:
doc=Document(
    page_content="this is the main text content I am using to create RAG pipeline",
    metadata={
        "source":"example.txt",
        "page":1,
        "author":"MD Ali",
        "date_created":"2026-04-02"
    }
)
doc

Document(metadata={'source': 'example.txt', 'page': 1, 'author': 'MD Ali', 'date_created': '2026-04-02'}, page_content='this is the main text content I am using to create RAG pipeline')

In [3]:
## Create a simple text file
import os
os.makedirs("../data/text_file",exist_ok=True)

In [4]:

sample_texts={
    "../data/text_file/python_intro.txt":"""Python Programming Introduction
    Python is a high-level, interpreted programming language that is easy to learn and use.
    It is widely used for web development, data analysis, artificial intelligence, and more.

    Key Features:
    - Easy to read and write
    - Strong community support
    - Versatile and powerful
    
    Python is widely used in various fields including:
    - Web development
    - Data analysis
    - Artificial Intelligence
    - Scientific computing
    """,
    "../data/text_file/machine_learning.txt":"""Machine Learning Basics
    Machine Learning is a field of artificial intelligence that focuses on building systems that learn from data.
    It involves training models to make predictions or decisions based on input data.

    Key Concepts:
    - Supervised Learning
    - Unsupervised Learning
    - Reinforcement Learning

    Machine Learning is used in various applications such as:
    - Image classification
    - Natural Language Processing
    - Recommendation Systems
    """
}

for filepath,content in sample_texts.items():
    with open(filepath,"w",encoding="utf-8") as f:
        f.write(content)
print("Text files created successfully")

Text files created successfully


In [5]:
### TextLoader
from langchain_community.document_loaders import TextLoader

In [6]:
loader=TextLoader("../data/text_file/python_intro.txt",encoding="utf-8")
documents=loader.load()
print(documents)

[Document(metadata={'source': '../data/text_file/python_intro.txt'}, page_content='Python Programming Introduction\n    Python is a high-level, interpreted programming language that is easy to learn and use.\n    It is widely used for web development, data analysis, artificial intelligence, and more.\n\n    Key Features:\n    - Easy to read and write\n    - Strong community support\n    - Versatile and powerful\n\n    Python is widely used in various fields including:\n    - Web development\n    - Data analysis\n    - Artificial Intelligence\n    - Scientific computing\n    ')]


In [7]:
from langchain_community.document_loaders import DirectoryLoader
dir_loader=DirectoryLoader(
    "../data/text_file",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding":"utf-8"},
    show_progress=False
    )
documents=dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_file\\machine_learning.txt'}, page_content='Machine Learning Basics\n    Machine Learning is a field of artificial intelligence that focuses on building systems that learn from data.\n    It involves training models to make predictions or decisions based on input data.\n\n    Key Concepts:\n    - Supervised Learning\n    - Unsupervised Learning\n    - Reinforcement Learning\n\n    Machine Learning is used in various applications such as:\n    - Image classification\n    - Natural Language Processing\n    - Recommendation Systems\n    '),
 Document(metadata={'source': '..\\data\\text_file\\python_intro.txt'}, page_content='Python Programming Introduction\n    Python is a high-level, interpreted programming language that is easy to learn and use.\n    It is widely used for web development, data analysis, artificial intelligence, and more.\n\n    Key Features:\n    - Easy to read and write\n    - Strong community support\n    - Versatile and p

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)
chunks = text_splitter.split_documents(documents)
print(len(chunks))

4


In [9]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

C:\Users\mdali\AppData\Local\Temp\ipykernel_15740\494109701.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(chunks, embeddings)

In [ ]:
retriever = vectorstore.as_retriever()

In [ ]:
# Load documents
documents = dir_loader.load()

# Split
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Chunks:", len(chunks))


# Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)


# Vector DB
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)


# Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


# Query
query = "What are key features of Python?"

results = retriever.invoke(query)

for i, r in enumerate(results):
    print(f"\nResult {i+1}:")
    print(r.page_content)

Chunks: 8


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1589.85it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Result 1:
Python is widely used in various fields including:
    - Web development
    - Data analysis
    - Artificial Intelligence
    - Scientific computing

Result 2:
Python Programming Introduction
    Python is a high-level, interpreted programming language that is easy to learn and use.

Result 3:
Key Features:
    - Easy to read and write
    - Strong community support
    - Versatile and powerful
